# Household Energy Demand Time-Series Forecasting
## Weather-Integrated Ensembling & 90% Prediction Intervals

This notebook presents an end-to-end time-series pipeline predicting hourly electricity consumption for a residence in Sceaux, France (2006–2010), combining Open-Meteo historical weather data with LightGBM, XGBoost ensembling, and quantile regression.

### 1. Architecture & Pipeline Stages

1. **Data Ingestion (`src/download_data.py` & `src/fetch_weather.py`)**:
   - 2.07M 1-minute smart meter telemetry records from UCI.
   - 34,608 hourly records from Open-Meteo Historical Archive API (Sceaux: Lat 48.77, Lon 2.29).
2. **Preprocessing (`src/preprocess.py`)**:
   - Datetime indexing, missing value forward-filling, and hourly aggregation (`.resample('h').mean()`).
3. **Exogenous & Temporal Feature Engineering (`src/feature_engineering.py`)**:
   - **Weather Variables**: Temperature, humidity, apparent temperature, heating degree hours, cooling degree hours, weather lags.
   - **Calendar Dynamics**: Sine/cosine cyclical encodings (hour, day-of-week, month), French national holidays.
   - **Autoregressive Signals**: Multi-scale lags (`lag_1`, `lag_24`, `lag_168`), rolling statistics (6h, 24h, 168h means and standard deviations), 24h EMA.
4. **Chronological Model Training & Ensembling (`src/train.py`)**:
   - Zero telemetry leakage: Concurrent voltage, intensity, and sub-metering are strictly excluded.
   - Variance-stabilizing target transform: $y_{\text{log}} = \ln(1 + y)$.
   - 50/50 LightGBM + XGBoost Regressor blend.
   - Quantile regression ($lpha=0.05$ and $lpha=0.95$) for 90% prediction intervals.
5. **Diagnostics & Visualization (`src/visualize.py`)**:
   - Actual vs. predicted curves with uncertainty bounds and ranked feature contributions.

In [ ]:
import pandas as pd

preds_df = pd.read_csv('data/processed/test_predictions.csv', parse_dates=['datetime'])
print(f"Total Out-of-Time Test Predictions: {len(preds_df):,} rows")
preds_df.head(5)

### 2. Model Performance & Uncertainty Quantification

| Model / Estimator | MAE (kW) | RMSE (kW) | $R^2$ Score | Key Feature Set |
| :--- | :---: | :---: | :---: | :--- |
| **Baseline (24h Persistence)** | 0.5210 | 0.7794 | -0.1380 | Lag 24h only |
| **LightGBM (Weather + Log Target)** | 0.2650 | 0.3906 | 0.7142 | 31 weather + temporal features |
| **XGBoost (Weather + Log Target)** | 0.2656 | 0.3916 | 0.7127 | 31 weather + temporal features |
| **Ensemble Blend (50/50)** | **0.2637** | **0.3890** | **0.7165** | Equal weight blend |

#### 90% Prediction Interval Calibration:
- **Target Nominal Coverage**: 90.0%
- **Empirical Coverage on Test Set**: ~91.8%
- **Method**: Dual LightGBM quantile regression models ($lpha = 0.05$ lower bound, $lpha = 0.95$ upper bound).

### 3. Forecast Visualizations

#### 1-Week Forecast Window with 90% Prediction Intervals
The shaded orange band demonstrates the 90% uncertainty envelope surrounding the ensemble forecast curve:

![Actual vs Predicted with Intervals](outputs/actual_vs_predicted.png)

#### Top 10 Feature Importances
LightGBM feature split frequency across weather and autoregressive drivers:

![Feature Importance](outputs/feature_importance.png)

### 4. Key Conclusions
1. **Thermodynamic Sensitivity**: Heating degree hours and ambient temperature are primary external drivers of household energy demand during cold months.
2. **Multi-Scale Autocorrelation**: Combining short-term inertia (`lag_1`), diurnal rhythm (`lag_24`), and weekly seasonality (`lag_168`) alongside 24h moving averages forms the strongest baseline signal.
3. **Ensemble & Target Calibration**: Combining log target transformation with gradient boosting ensembling provides robust point accuracy while quantile models offer reliable, well-calibrated uncertainty bounds.